# 01 — Data Acquisition

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Programmatically download or load the raw datasets (SEC EDGAR, EPA GHGRP, World Bank).

---


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest_sec import SECIngester
from src.ingest_epa import EPAIngester
from src.ingest_worldbank import WorldBankIngester

RAW_DIR = PROJECT_ROOT / "data" / "raw"
os.makedirs(RAW_DIR, exist_ok=True)

print("Project Root:", PROJECT_ROOT)

Project Root: E:\Research_Projects\predicting-corporate-ghg-intensity


### 1. Ingest EPA GHGRP Emissions
We fetch facility-level emissions. If real data is cached, we load it directly.

In [2]:
if (RAW_DIR / "epa_ghgrp_facilities.csv").exists():
    df_epa = pd.read_csv(RAW_DIR / "epa_ghgrp_facilities.csv")
    print("Loaded cached real EPA GHGRP data.")
else:
    epa_ingester = EPAIngester(output_dir=RAW_DIR)
    df_epa = epa_ingester.fetch_epa_emissions(force_simulate=True)
print("EPA Shape:", df_epa.shape)
df_epa.head()

Loaded cached real EPA GHGRP data.
EPA Shape: (39230, 11)


,facility_id,facility_name,reported_parent,year,state,total_ghg_emissions,co2_emissions_non_biogenic,primary_naics_code,naics_2digit,naics_sector,high_emission_naics
0,1004377,121 REGIONAL DISPOSAL FACILITY,NORTH TEXAS MUNICIPAL WATER DISTRICT,2018,TX,653854.000,NaN,562212.0,56.0,Administrative,0
1,1010040,15-18565/15-18662,CAMBRIAN COAL LLC,2018,KY,125981.750,NaN,212112.0,21.0,Mining,1
2,1010085,15-19015,CAMBRIAN COAL LLC,2018,KY,93918.750,NaN,212112.0,21.0,Mining,1
3,1001155,1500 South Tibbs LLC d/b/a Aurorium Indianapol...,VERTELLUS HOLDINGS LLC,2018,IN,72163.662,72064.6,325199.0,32.0,Manufacturing,1
4,1000112,23rd and 3rd,NEW YORK POWER AUTHORITY,2018,NY,70705.288,70633.5,221112.0,22.0,Utilities,1


### 2. Ingest SEC EDGAR Financials
We pull financials for the full registry of firms. If real data is cached, we load it directly.

In [3]:
if (RAW_DIR / "sec_financials.csv").exists():
    df_sec = pd.read_csv(RAW_DIR / "sec_financials.csv")
    print("Loaded cached real SEC EDGAR financials.")
else:
    sec_ingester = SECIngester(output_dir=RAW_DIR)
    df_sec = sec_ingester.fetch_sec_financials(force_simulate=True)
print("SEC Shape:", df_sec.shape)
df_sec.head()

Loaded cached real SEC EDGAR financials.
SEC Shape: (2791, 14)


,cik,ticker,company_name,sector,sic_code,year,total_assets,revenue,net_income,operating_income,capex,rd_expense,total_debt,stockholders_equity
0,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2018,33246625.0,37174249.0,2001185.0,2402648.0,NaN,0.0,NaN,29759749.0
1,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2019,31723376.0,32873002.0,538314.0,491584.0,NaN,0.0,NaN,29158027.0
2,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2020,31238071.0,27590653.0,50450.0,-83014.0,NaN,0.0,NaN,28706089.0
3,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2021,31766258.0,33974558.0,1113472.0,1358915.0,NaN,0.0,NaN,28969365.0
4,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2022,33626127.0,33646033.0,2867629.0,3561196.0,NaN,0.0,NaN,30986798.0


### 3. Ingest World Bank ESG Controls
We fetch US macro economic controls (GDP growth and emissions per capita).

In [4]:
if (RAW_DIR / "worldbank_macro.csv").exists():
    df_wb = pd.read_csv(RAW_DIR / "worldbank_macro.csv")
    print("Loaded cached real World Bank macro controls.")
else:
    wb_ingester = WorldBankIngester(output_dir=RAW_DIR)
    df_wb = wb_ingester.fetch_macro_controls(force_simulate=True)
print("World Bank Shape:", df_wb.shape)
df_wb.head()

Loaded cached real World Bank macro controls.
World Bank Shape: (6, 4)


,year,us_gdp_growth,us_co2_per_capita,us_energy_use_per_capita
0,2018,2.966505,15.2,6738.272724
1,2019,2.583825,14.8,6700.930340
2,2020,-2.163029,13.0,6141.531258
3,2021,6.055053,13.9,6445.834092
4,2022,2.512375,13.6,6511.688414


### Discussion & Next Steps
Raw datasets have been acquired and stored in `data/raw/` as CSV files. In the next notebook, we will run the entity resolution matching to link EPA parent company records with their corresponding SEC corporate CIKs.